In [ ]:
symbols = {"NVDA","META","BYND","CHGG","AAPL","BRK-B","JNJ","AMZN","GOOGL","TSLA","AMD","PLTR","RIVN","COIN","AI","UPST","FSLY","OPEN","HOOD","SPY","QQQ","XLF","XLE","ARKK","SPY"} # diverse data

import yfinance as yf
import functions
import pandas as pd
from PIL import Image
from IPython.display import display
import random
from datetime import datetime
import warnings
import math
# loop through symbols, use predictTest, use frequencies as seconds, use 1mo,3mo,6mo,1y,2y,5y as range

warnings.simplefilter(action='ignore', category=FutureWarning)
charts = functions.Charts()

def distribute(values:list, error:float, proximity:float):
    offset = error*proximity
    nudged = [v + random.uniform(-offset, offset) for v in values]
    nudged = [max(v, 0.001) for v in nudged]
    total = sum(nudged)
    return [v / total for v in nudged]

ranges = ["2022-01-01","2025-11-30"]
dates = pd.date_range(start=ranges[0], end=ranges[1])

In [91]:
biases:dict[str,list] = {}

for symbol in symbols:
    print(symbol)
    sector = yf.Sector(yf.Ticker(symbol).info.get("sectorKey")).ticker.info["displayName"]
    prices = yf.download(symbol, start=ranges[0], end=ranges[1], progress=False)["Close"]

    for i, date in enumerate(dates):
        bestWeight = [0.2,0.2,0.2,0.2,0.2] #baseline: "duration":[weight,freq] - freq in seconds
        bestError = 9999.0
        bestProx = 0.1
        trials = 0

        while trials <= 60:
            tests:list = distribute(bestWeight,bestError,bestProx)
            bias = {90:[float(tests[0]), "ME"], 180:[float(tests[1]), "ME"], 365:[float(tests[2]), "D"], 730:[float(tests[3]), "W"], 1825:[float(tests[4]), "YS"]}
            guess = round(charts.projectTestDay(ticker=symbol, weights=str(bias), today=date),2)
            actual = round(prices.iloc[i][0],2)
            error = abs(actual-guess)
            if error < bestError:
                bestWeight = tests
                bestError = error
                bestProx = bestError*0.01
            #print(trials, error, bestError, bestProx, tests, bestWeight)
            trials += 1
        if sector not in biases:
            biases[sector] = bestWeight
        else:
            prev = biases[sector]
            biases[sector] = [(float(prev[j] + bestWeight[j]) / 2) for j in range(len(prev))]
        print(biases)

NVDA
{'Technology': [np.float64(0.00100350901936868), np.float64(0.00100350901936868), np.float64(0.07203563133282381), np.float64(0.5479505089155808), np.float64(0.3780068417128579)]}
{'Technology': [0.003019154764818402, 0.0010073258707071707, 0.03660552003996978, 0.44752035554983977, 0.5118476437746649]}
{'Technology': [0.010282260589611927, 0.002122620627939628, 0.0839734493634156, 0.6453831962914918, 0.25823847312754106]}
{'Technology': [0.005641409852040831, 0.0299324671920073, 0.48603884233890265, 0.3478112868915326, 0.13057599372551662]}
{'Technology': [0.024771896841800718, 0.44393859894304155, 0.24640116332561302, 0.2191003052117473, 0.06578803567779741]}
{'Technology': [0.24786169985461748, 0.4433987772586937, 0.13521916759004413, 0.13065589699937563, 0.04286445829726901]}
{'Technology': [0.1288723557990291, 0.3654866654998191, 0.1491338758099794, 0.1632399585361219, 0.1932671443550505]}
{'Technology': [0.26281607235645915, 0.34802370436556085, 0.07580239518199042, 0.0823693

KeyboardInterrupt: 